In [1]:
!wget http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv


--2026-04-25 04:17:37--  http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv
Resolving cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)... 129.25.203.170
Connecting to cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)|129.25.203.170|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 302235399 (288M) [text/plain]
Saving to: ‘median.csv’

median.csv           68%[============>       ] 198.52M  2.09MB/s    eta 42s    ^C


In [2]:
!head -5 median.csv

yy,mm,dd,hh,mi,ss,seq,theTime,logID,clientIP,requestor,server,dbname,access,elapsed,busy,rows,statement,error,errorMessage,isvisible
2008,11,30,23,59,56,1767613780,11/30/2008 11:59:56 PM,8002,66.249.71.243,cas.sdss.org,SDSSSQL005,BestDR4,public,0.016,0,1,select * from Field where fieldId=0x08290cfd60860000  ,0,,1
2008,11,30,23,59,56,1767613779,11/30/2008 11:59:56 PM,8002,129.125.6.203,cas.sdss.org,SDSSSQL007,BestDR5,public,0.063,2E-3,431,"Select P.ra, P.dec, P.htmID, Q.objID,Q.photoz,Q.photozErr,Q.flag from PhotoObjAll P, Photoz2 Q where ((P.htmID >= 14504465268736 and P.htmID <= 14504469463039)) and P.objID = Q.objID  ",0,,1
2008,11,30,23,59,47,1767613781,11/30/2008 11:59:48 PM,8002,129.125.6.203,cas.sdss.org,SDSSSQL007,BestDR5,public,0.063,2E-3,338,"Select P.ra, P.dec, P.htmID, Q.objID,Q.photoz,Q.photozErr,Q.flag from PhotoObjAll P, Photoz2 Q where ((P.htmID >= 14504461074432 and P.htmID <= 14504465268735)) and P.objID = Q.objID  ",0,,1
2008,11,30,23,59,42,1767613786,11/30/2008 11:59

In [3]:
import pandas as pd
df = pd.read_csv("median.csv", on_bad_lines='warn',encoding='latin1')

/tmp/ipykernel_110/1631936815.py:2: ParserWarning: Skipping line 17948: expected 21 fields, saw 28

  df = pd.read_csv("median.csv", on_bad_lines='warn',encoding='latin1')
/tmp/ipykernel_110/1631936815.py:2: ParserWarning: Skipping line 146581: expected 21 fields, saw 23
Skipping line 146839: expected 21 fields, saw 23
Skipping line 147106: expected 21 fields, saw 23
Skipping line 147125: expected 21 fields, saw 23
Skipping line 147183: expected 21 fields, saw 23
Skipping line 147193: expected 21 fields, saw 23

  df = pd.read_csv("median.csv", on_bad_lines='warn',encoding='latin1')
/tmp/ipykernel_110/1631936815.py:2: ParserWarning: Skipping line 520687: expected 21 fields, saw 23

  df = pd.read_csv("median.csv", on_bad_lines='warn',encoding='latin1')
/tmp/ipykernel_110/1631936815.py:2: ParserWarning: Skipping line 548321: expected 21 fields, saw 50

  df = pd.read_csv("median.csv", on_bad_lines='warn',encoding='latin1')
/tmp/ipykernel_110/1631936815.py:2: ParserWarning: Skipping line

In [4]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [5]:
df.isna().sum(axis=1).value_counts().sort_index()

0      48983
1     777625
2          4
3       2319
4        138
12         1
13         1
14        15
15         5
16        15
17      1324
18      1235
19       116
20      6415
Name: count, dtype: int64

We are removing those which we have more than 2 null values because those are the querries where parsing have been failed we can reconstruct them by more careful inspection which we have tried but the amount of data we are getting back doesnt feel worth the work

In [6]:
df_cleaned=df[df.isna().sum(axis=1)<2]

In [7]:
colns=list(df_cleaned['error'].unique())

In [8]:
colns

[0.0,
 -1.0,
 '0',
 '-1',
 ' dec',
 '1',
 'dec FROM Star"',
 1.0,
 " s2.ew/(1+s2.z) as 'ew/(1+z)'",
 ' s2.lineID']

In [9]:
colns.remove(' dec')
colns.remove('dec FROM Star"')
colns.remove(" s2.ew/(1+s2.z) as 'ew/(1+z)'")
colns.remove(' s2.lineID')

In [10]:
df_cleaned=df_cleaned[df_cleaned['error'].isin(colns)]

In [11]:
# df_cleaned.errorMessage.unique()

We got the cleaned data 

In [12]:
import numpy as np
df_cleaned['errorMessage'].value_counts()

errorMessage
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [13]:
df_cleaned['error'] = pd.to_numeric(df_cleaned['error'], errors='coerce')

In [14]:
df_correct = df_cleaned[df_cleaned['error'] == 0]
print(df_correct.shape)

(817412, 21)


In [15]:
df_cleaned[df.error==1].sample(10)

/tmp/ipykernel_110/3752742740.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_cleaned[df.error==1].sample(10)


,yy,mm,dd,hh,mi,ss,seq,theTime,logID,clientIP,requestor,server,dbname,access,elapsed,busy,rows,statement,error,errorMessage,isvisible
714759,2008,11,5,16,15,59,1764214413,11/5/2008 4:15:59 PM,8002,200.14.68.146,,DR7Targ,TargDR7,casjobs,22.500,0.0,99999999.0,"select t.objid,t.ra,t.dec\ninto mydb.help2\nfr...",1.0,,1.0
713769,2008,11,5,17,23,13,1764213445,11/5/2008 5:23:14 PM,8002,144.118.201.119,,SQL010DBHost,MyDB,casjobs,0.030,0.0,99999999.0,select top 0 * from QWG.gtr.[yannytest2allnext...,1.0,,1.0
402411,2008,11,16,16,51,11.0,1767279803.0,11/16/2008 4:51:11 PM,8002.0,132.229.224.39,,sdsssql010,MYDB,casjobs,0.000,0.0,99999999.0,select * from Lynga7,1.0,,1.0
206428,2008,11,21,11,4,23.0,1767100660.0,11/21/2008 11:04:23 AM,8002.0,130.183.74.206,,sdsssql010,MYDB,casjobs,0.000,0.0,99999999.0,select * from tab1795,1.0,,1.0
539543,2008,11.0,13,9.0,31.0,7.0,1767404516.0,11/13/2008 9:31:07 AM,8002.0,130.183.74.104,,DR7Best long,BestDR7,casjobs,13.906,0.0,99999999.0,"select p.ra, p.dec, q.z, q.zErr into mydb.MyTa...",1.0,,1.0
713765,2008,11,5,17,23,16,1764213442,11/5/2008 5:23:16 PM,8002,144.118.201.119,,SQL010DBHost,MyDB,casjobs,0.030,0.0,99999999.0,select top 0 * from QWG.gtr.[yannytest2allnext...,1.0,,1.0
199179,2008,11,21,16,36,21.0,1767096407.0,11/21/2008 4:36:22 PM,8002.0,131.225.7.9,,DR7Collab,BestDR7Collab,casjobs,4.373,0.0,99999999.0,"select isnull(ph.run,sp.brun) as run,isnull(...",1.0,,1.0
206476,2008,11,21,11,2,23.0,1767100710.0,11/21/2008 11:02:23 AM,8002.0,130.183.74.206,,sdsssql010,MYDB,casjobs,0.016,0.0,99999999.0,select * from tab1795,1.0,,1.0
665665,2008,11,7,15,10,35.0,1764167477.0,11/7/2008 3:10:35 PM,8002.0,129.59.116.15,,DR7Best,BestDR7,casjobs,0.080,0.0,99999999.0,"CREATE TABLE #upload (up_ra FLOAT, up_dec FLOA...",1.0,,1.0
529613,2008,11.0,13,14.0,25.0,0.0,1767401212.0,11/13/2008 2:25:00 PM,8002.0,128.95.99.118,,Stripe82 long,Stripe82,casjobs,0.063,0.0,99999999.0,"select s.run, s.camcol, s.field, s.flags_u, s....",1.0,,1.0


In [16]:
statements=list(df_correct.statement)

In [17]:
import re
import sqlparse
from sqlparse.tokens import Number, String, Literal

def normalize_sql(sql):
    """Lowercase + collapse all whitespace into single spaces"""
    if not isinstance(sql, str):
        return ""
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)   
    return sql

def tokenize_sql(sql):
    """Robustly tokenize SQL and mask literals in one pass to prevent fragmentation"""
    sql = normalize_sql(sql)
    if not sql:
        return []
        
    parsed = sqlparse.parse(sql)
    if not parsed:
        return []
        
    tokens = []
    for token in parsed[0].flatten():
        if token.is_whitespace:
            continue
            
        if token.ttype in Number:
            tokens.append('NUM_LITERAL')
        elif token.ttype in String:
            tokens.append('STR_LITERAL')
        elif token.value.startswith('0x') and len(token.value) > 2:
            tokens.append('HEX_LITERAL')
        else:
            tokens.append(token.value)
            
    return tokens

In [18]:
# This function is now legacy as the logic was moved into tokenize_sql for robustness.
def mask_literals(sql):
    return " ".join(tokenize_sql(sql))

In [19]:
sql = "select p.ra, p.dec from photoobjall p where p.htmid >= 14504465268736"
tokens = tokenize_sql(sql)
print(tokens)
print("scientfic literal check:", tokenize_sql("SELECT busy FROM logs WHERE busy > 2E-3"))

['select', 'p', '.', 'ra', ',', 'p', '.', 'dec', 'from', 'photoobjall', 'p', 'where', 'p', '.', 'htmid', '>=', 'NUM_LITERAL']
scientfic literal check: ['select', 'busy', 'from', 'logs', 'where', 'busy', '>', 'NUM_LITERAL']


In [20]:
!git clone https://github.com/SQL-Storm/SQLStorm.git

Cloning into 'SQLStorm'...
remote: Enumerating objects: 462114, done.
remote: Counting objects: 100% (1312/1312), done.
remote: Compressing objects: 100% (1279/1279), done.
remote: Total 462114 (delta 137), reused 33 (delta 33), pack-reused 460802 (from 5)
Receiving objects: 100% (462114/462114), 328.36 MiB | 23.79 MiB/s, done.
Resolving deltas: 100% (83187/83187), done.
Updating files: 100% (586415/586415), done.
Filtering content: 100% (82/82), 657.88 MiB | 23.35 MiB/s, done.


In [21]:
import os 
os.chdir("SQLStorm")

In [22]:
ls v1.0/tpch

distinct_queries.csv  queries_generated/  queries.sql
queries/              queries_snowflake/


In [23]:
import pandas as pd
import os


def load_sql_dataframes(base_path):
    """
    Reads valid and invalid SQL query CSVs from the SQLStorm dataset,
    loads the corresponding .sql files, and returns two labeled DataFrames.

    Parameters
    ----------
    base_path : str
        Root path to the SQLStorm dataset (e.g., "/kaggle/working/SQLStorm/").

    Returns
    -------
    df_valid : pd.DataFrame
        DataFrame with columns ['file_name', 'sql_text', 'label'] where label=0.
    df_invalid : pd.DataFrame
        DataFrame with columns ['file_name', 'sql_text', 'label'] where label=1.
    """

    valid_csv_path = os.path.join(base_path, "v1.0", "stackoverflow", "valid_queries.csv")
    valid_queries_dir = os.path.join(base_path, "v1.0", "stackoverflow", "queries")

    valid_csv = pd.read_csv(valid_csv_path)
    valid_file_names = []
    valid_sql_texts = []

    for _, row in valid_csv.iterrows():
        file_name = str(row.iloc[0]) 
        sql_file_path = os.path.join(valid_queries_dir, file_name)

        try:
            with open(sql_file_path, "r", encoding="utf-8") as f:
                sql_text = f.read()
            valid_file_names.append(file_name)
            valid_sql_texts.append(sql_text)
        except FileNotFoundError:
            print(f"[SKIP] Valid query file not found: {sql_file_path}")
            continue

    invalid_csv_path = os.path.join(base_path, "v1.0", "stackoverflow", "invalid_queries.csv")
    invalid_queries_dir = os.path.join(base_path, "v1.0", "stackoverflow", "queries_generated")

    invalid_csv = pd.read_csv(invalid_csv_path)
    invalid_file_names = []
    invalid_sql_texts = []

    for _, row in invalid_csv.iterrows():
        file_name = str(row.iloc[0]) 
        sql_file_path = os.path.join(invalid_queries_dir, file_name)

        try:
            with open(sql_file_path, "r", encoding="utf-8") as f:
                sql_text = f.read()
            invalid_file_names.append(file_name)
            invalid_sql_texts.append(sql_text)
        except FileNotFoundError:
            print(f"[SKIP] Invalid query file not found: {sql_file_path}")
            continue

    df_valid = pd.DataFrame({
        "file_name": valid_file_names,
        "sql_text": valid_sql_texts,
        "label": 0
    })

    df_invalid = pd.DataFrame({
        "file_name": invalid_file_names,
        "sql_text": invalid_sql_texts,
        "label": 1
    })

    print(f"Loaded {len(df_valid)} valid queries   (label=0)")
    print(f"Loaded {len(df_invalid)} invalid queries (label=1)")

    return df_valid, df_invalid


In [24]:
df_valid_stack_overflow,df_invalid_stack_overflow=load_sql_dataframes("/kaggle/working/SQLStorm")

Loaded 12587 valid queries   (label=0)
Loaded 5664 invalid queries (label=1)


In [25]:
import pandas as pd
import os


def load_sql_dir(directory):
    """
    Load every .sql file in `directory` into a pandas DataFrame.

    Parameters
    ----------
    directory : str
        Path to the folder containing .sql files.

    Returns
    -------
    pd.DataFrame
        Columns: ['file_name', 'sql_text']
    """
    records = []

    for file_name in sorted(os.listdir(directory)):
        if file_name.lower().endswith(".sql"):
            file_path = os.path.join(directory, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    sql_text = f.read()
                records.append({"file_name": file_name, "sql_text": sql_text})
            except (FileNotFoundError, PermissionError, UnicodeDecodeError) as e:
                print(f"[SKIP] {file_path} → {e}")

    df = pd.DataFrame(records, columns=["file_name", "sql_text"])
    print(f"Loaded {len(df)} .sql files from {directory}")
    return df
df_tpch=load_sql_dir("/kaggle/working/SQLStorm/v1.0/tpch/queries")

Loaded 17036 .sql files from /kaggle/working/SQLStorm/v1.0/tpch/queries


In [26]:
df_tpcds=load_sql_dir("/kaggle/working/SQLStorm/v1.0/tpcds/queries")

Loaded 15242 .sql files from /kaggle/working/SQLStorm/v1.0/tpcds/queries


In [27]:
df_job=load_sql_dir("/kaggle/working/SQLStorm/v1.0/job/queries")

Loaded 11714 .sql files from /kaggle/working/SQLStorm/v1.0/job/queries


In [28]:
df_sqlstorm=pd.concat([df_valid_stack_overflow,df_tpcds,df_tpch,df_job],axis=0)

In [29]:
df_sqlstorm.drop('label',axis=1,inplace=True)

In [ ]:
print(df_sqlstorm.shape)

(56579, 2)


In [30]:
statements=statements[:50000]+list(df_sqlstorm['sql_text'])

In [32]:
import pandas as pd

# Clean
statements = [q for q in statements if isinstance(q, str) and q.strip()]

# Save
with open("sql_queries_1.txt", "w", encoding="utf-8") as f:
    for q in statements:
        q = q.strip().replace("\n", " ")  # remove internal newlines
        f.write(f"!#@{q}#$@\n")

print(f"Saved {len(statements)} queries with custom delimiters")

Saved 106579 queries with custom delimiters


In [ ]:
print(statements[1200])

Select P.ra, P.dec, P.htmID, Q.objID,Q.photoz,Q.photozErr,Q.flag from PhotoObjAll P, Photoz2 Q where ((P.htmID >= 14501885771776 and P.htmID <= 14501889966079)) and P.objID = Q.objID  


In [ ]:
print(statements[60000])


SELECT 
    p.Id AS PostId,
    p.Title,
    u.DisplayName AS Owner,
    p.CreationDate,
    p.Score,
    p.ViewCount,
    COUNT(c.Id) AS CommentCount
FROM 
    Posts p
JOIN 
    Users u ON p.OwnerUserId = u.Id
LEFT JOIN 
    Comments c ON p.Id = c.PostId
WHERE 
    p.PostTypeId = 1  
GROUP BY 
    p.Id, p.Title, u.DisplayName, p.CreationDate, p.Score, p.ViewCount
ORDER BY 
    p.CreationDate DESC
LIMIT 10;



In [2]:
def load_queries(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    statements = content.split("#$@")
    statements = [q.replace("!#@", "").strip() for q in statements if q.strip()]
    return statements

In [3]:
statements_loaded = load_queries("/kaggle/input/datasets/balasiva2006/dbms-project-workload-summarization/dbms_projectt - Copy/generated_data/sql_queries.txt")

In [4]:

def save_queries(file_path, queries):
    with open(file_path, "w", encoding="utf-8") as f:
        for q in queries:
            f.write(f"!#@{q}#$@")   # same format


In [5]:
import random
random.seed(42)
sample_size = min(7000, len(statements_loaded))
sampled_queries = random.sample(statements_loaded, sample_size)

# Save sampled queries
output_path = "sampled_queries_2.txt"
save_queries(output_path, sampled_queries)

print(f"Saved {len(sampled_queries)} queries to {output_path}")

Saved 7000 queries to sampled_queries_2.txt
